<a href="https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content piece (webpage) with aggregated 90-day performance metrics.

No time column exists. All metrics are snapshots of the last 90 days or last 30 days.

I will predict trend_direction (down vs not down) from current snapshot features.

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/content_refresh_anonymized.csv')

# Check the structure
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 3 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (knowable at decision time):
- word_count: page content length, known before prediction
- page_age_days: how old the page is, known from creation date
- impressions_90d: past 90 days traffic, historical
- clicks_90d: past 90 days clicks, historical
- days_since_update: time since last content update

LABEL (what we predict):
- trend_direction: 'down' vs 'not down' — is this page declining?

CONTEXT (not used as features, but useful to know):
- month: which month this row represents
- page_id: anonymous identifier

EXCLUDED (and why):
- page_url: unique identifier, would cause overfitting, not a real feature
- Any column with >50% missing values: too unreliable

In [3]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

Missing values per column:
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_orde

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts to prove:
1. Grain: one row = one page per month
2. Count: how many rows in 2026-03
3. Availability: how many rows have complete data

In [12]:
# ---------- LEAKAGE EXPERIMENT ----------
print("\n" + "=" * 50)
print("LEAKAGE EXPERIMENT: The Trap Everyone Falls Into")
print("=" * 50)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Prepare target
y = (df['trend_direction'] == 'down').astype(int)

# Clean features (no leakage) - fill NaN with 0
X_clean = df[['word_count', 'content_age_days', 'impressions_90d', 'clicks_90d', 'ctr']].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

model_clean = LogisticRegression(max_iter=1000)
model_clean.fit(X_train, y_train)
auc_clean = roc_auc_score(y_test, model_clean.predict_proba(X_test)[:, 1])

print(f"\nAUC WITHOUT leakage (honest): {auc_clean:.3f}")

# Now add leakage: trend_pct is derived from the label - FILL NA WITH 0
X_leak = X_clean.copy()
X_leak['trend_pct'] = df['trend_pct'].fillna(0)  # <-- FIXED HERE

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.2, random_state=42, stratify=y
)

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train_l, y_train_l)
auc_leak = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])

print(f"AUC WITH leakage (fake):    {auc_leak:.3f}")
print(f"\nLeakage boost: +{auc_leak - auc_clean:.3f}")
print("This boost is FAKE. trend_pct is just the label in numeric form.")
print("I delete it and keep the honest number.")

# Delete leakage column
del X_leak


LEAKAGE EXPERIMENT: The Trap Everyone Falls Into

AUC WITHOUT leakage (honest): 0.601
AUC WITH leakage (fake):    1.000

Leakage boost: +0.399
This boost is FAKE. trend_pct is just the label in numeric form.
I delete it and keep the honest number.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits:

1. HIGH IMBALANCE: 54.2% declining vs 45.8% other. Model must beat 54% baseline.

2. MISSING WORD_COUNT: 25.7% of rows lack word_count. I filled with 0, but this may bias results.

3. NO TIME DIMENSION: Cannot verify if trends are seasonal, accelerating, or stable. Cross-sectional only.

4. CROSS-SECTIONAL: All metrics from same 90-day window. Cannot establish causation.

5. LABEL QUALITY: "stable", "flat", "new" categories may overlap or be inconsistently defined.

6. LEAKAGE RISK: trend_pct is derived from trend_direction. Must exclude from features.

In [9]:
# Check label distribution
print("Trend direction counts:")
print(df['trend_direction'].value_counts())

print(f"\nDeclining rate: {(df['trend_direction'] == 'down').mean()*100:.1f}%")

# Check missing values in key features
print("\nMissing values in features:")
features = ['word_count', 'content_age_days', 'days_since_last_update',
            'impressions_90d', 'clicks_90d', 'ctr', 'avg_position']
print(df[features].isnull().sum())

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%

Missing values in features:
word_count                7699
content_age_days             0
days_since_last_update       0
impressions_90d              0
clicks_90d                   0
ctr                          0
avg_position                 0
dtype: int64
